<div style="background-color: lightblue; color: black; padding: 10px; border-radius: 5px;">

## Mini Project: Exploring LLMs using Hugging Face API (Week 33)

###  Objective
This notebook demonstrates how to leverage a pre-trained Language Model from Hugging Face to build a simple Question & Answering (Q&A) system. The goal is to create a core engine that can accept any question and a context (like a paragraph of text) and extract the correct answer from that context. This prototype serves as the initial step towards developing a full-fledged chatbot application.
</div>

<div style="background-color: lightblue; color: black; padding: 10px; border-radius: 5px;">

### Load Libraries
</div>

In [ ]:
import json
import os
from datasets import Dataset
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset




<div style="background-color: lightblue; color: black; padding: 10px; border-radius: 5px;">

### Data Generation - Create Training Dataset
</div>

In [3]:
# Cell 1: Data Generation - Create Training Dataset
"""
This cell generates 500+ high-quality question-answer pairs about machine learning
and deep learning topics. The data is saved in JSONL format for training.
"""

import json
import os

ML_DL_DATA = [
    {
        "instruction": "What is machine learning and how does it differ from traditional programming?",
        "input": "",
        "output": "Machine learning is a subset of AI where systems learn from data rather than following explicit rules. Traditional programming uses fixed rules, while ML learns patterns from data automatically."
    },
    {
        "instruction": "Explain supervised, unsupervised, and reinforcement learning",
        "input": "",
        "output": "Supervised learning: learns from labeled examples. Unsupervised learning: finds patterns in unlabeled data. Reinforcement learning: learns through trial and error with rewards."
    },
    {
        "instruction": "What is the bias-variance tradeoff?",
        "input": "",
        "output": "Bias is error from oversimplifying assumptions. Variance is error from being too sensitive to training data. The goal is to find the sweet spot where total error is minimized."
    },
    {
        "instruction": "Explain gradient descent",
        "input": "",
        "output": "Gradient descent is an optimization algorithm that finds the minimum of a function by iteratively moving in the direction of steepest descent (negative gradient)."
    },
    {
        "instruction": "What are learning rates and how do you choose them?",
        "input": "",
        "output": "Learning rates control step size in optimization. Common values: 0.001 for Adam, 0.01 for SGD. Use learning rate finders to find optimal range."
    },
    {
        "instruction": "What is a neural network and how does it work?",
        "input": "",
        "output": "A neural network consists of interconnected layers of artificial neurons. Data flows forward, each neuron computes y = activation(sum(weights * inputs) + bias), and backpropagation adjusts weights."
    },
    {
        "instruction": "Explain Convolutional Neural Networks (CNNs)",
        "input": "",
        "output": "CNNs use convolution operations to detect features in images. Applications: image classification, object detection, facial recognition, and medical imaging."
    },
    {
        "instruction": "What are Recurrent Neural Networks (RNNs)?",
        "input": "",
        "output": "RNNs process sequential data with recurrent connections that maintain a hidden state. Use for NLP, time series, speech recognition, and machine translation."
    },
    {
        "instruction": "What are Transformers and why are they revolutionary?",
        "input": "",
        "output": "Transformers use self-attention mechanisms to process sequences in parallel. They handle long-range dependencies better and are the foundation of modern NLP (GPT, BERT)."
    },
    {
        "instruction": "What are activation functions and why are they needed?",
        "input": "",
        "output": "Activation functions introduce non-linearity. Common ones: ReLU (most popular), Sigmoid (binary classification), Tanh (hidden layers), Softmax (multi-class output)."
    },
    {
        "instruction": "Explain different loss functions",
        "input": "",
        "output": "Regression: MSE (sensitive), MAE (robust). Classification: Binary Cross-Entropy, Categorical Cross-Entropy. Rule of thumb: Regression -> MSE, Classification -> Cross-entropy."
    },
    {
        "instruction": "What is regularization and why is it important?",
        "input": "",
        "output": "Regularization prevents overfitting. Techniques: L1/L2 regularization, Dropout, Early Stopping, Data Augmentation, and Batch Normalization."
    },
    {
        "instruction": "What are evaluation metrics for machine learning?",
        "input": "",
        "output": "Classification: Accuracy, Precision, Recall, F1-Score, ROC-AUC. Regression: MAE, MSE, RMSE, R^2. For imbalanced data, use precision, recall, and F1-score."
    },
    {
        "instruction": "What is tokenization in NLP?",
        "input": "",
        "output": "Tokenization converts text into tokens. Types: Word (simple), Character (no unknown words), Subword (best balance, used in BPE, WordPiece, SentencePiece)."
    },
    {
        "instruction": "What are Large Language Models (LLMs)?",
        "input": "",
        "output": "LLMs are neural networks with billions of parameters trained on massive text datasets using Transformer architecture. Popular models: GPT, LLaMA, Claude, Gemini, Mistral."
    },
    {
        "instruction": "What is the workflow for a machine learning project?",
        "input": "",
        "output": "1. Problem Definition, 2. Data Collection, 3. Data Preparation, 4. EDA, 5. Feature Engineering, 6. Model Selection, 7. Training, 8. Evaluation, 9. Deployment."
    },
    {
        "instruction": "What is cross-validation?",
        "input": "",
        "output": "Cross-validation evaluates model performance by splitting data into multiple subsets. Types: K-Fold (most common), Stratified K-Fold (for imbalanced data)."
    },
    {
        "instruction": "How do you handle overfitting?",
        "input": "",
        "output": "Strategies: 1. More Data, 2. Regularization, 3. Simpler Architecture, 4. Early Stopping, 5. Reduce Learning Rate, 6. Transfer Learning."
    },
    {
        "instruction": "What is transfer learning?",
        "input": "",
        "output": "Transfer learning applies knowledge from one task to a related task. Approaches: Feature Extraction, Fine-Tuning, Multi-task Learning. Powerful with limited data."
    },
    {
        "instruction": "What are GANs and how do they work?",
        "input": "",
        "output": "GANs consist of Generator (creates fake data) and Discriminator (distinguishes real from fake). They improve together until generator produces realistic data."
    },
    {
        "instruction": "What is the attention mechanism?",
        "input": "",
        "output": "Attention allows models to focus on relevant parts of input. Types: Self-attention, Cross-attention. Advantages: parallel processing, better long-range dependencies."
    },
    {
        "instruction": "What are word embeddings?",
        "input": "",
        "output": "Word embeddings are dense vectors capturing semantic meaning. Techniques: Word2Vec, GloVe, FastText, BERT. Benefits: lower dimensionality, semantic relationships."
    },
    {
        "instruction": "What is the ReLU activation function?",
        "input": "",
        "output": "ReLU (Rectified Linear Unit): f(x) = max(0, x). Popular because: computationally efficient, sparse activation, no vanishing gradient. Issue: dead neurons for negative values."
    },
    {
        "instruction": "What are optimization algorithms in deep learning?",
        "input": "",
        "output": "SGD (fast but noisy), Momentum (accelerates), Adam (most popular, combines momentum and RMSprop), AdamW (decoupled weight decay)."
    },
    {
        "instruction": "Explain the Adam optimizer",
        "input": "",
        "output": "Adam adapts learning rates per parameter using first and second moments of gradients. Default: lr=0.001, beta1=0.9, beta2=0.999. Advantages: efficient, works with large datasets."
    },
    {
        "instruction": "What is dropout and how does it work?",
        "input": "",
        "output": "Dropout randomly drops neurons during training (p=0.5). Prevents co-adaptation, forces redundant representations, and acts as model averaging."
    },
    {
        "instruction": "What is backpropagation?",
        "input": "",
        "output": "Backpropagation calculates gradients of the loss function with respect to all weights using the chain rule. It's the core algorithm for training neural networks."
    },
    {
        "instruction": "What is the difference between CNN and RNN?",
        "input": "",
        "output": "CNNs detect spatial patterns in images (grid-like data). RNNs process sequential data (time series, text) by maintaining hidden states. CNNs are feed-forward, RNNs have recurrent connections."
    },
    {
        "instruction": "What is batch normalization?",
        "input": "",
        "output": "Batch normalization normalizes layer inputs, reducing internal covariate shift. It stabilizes training, allows higher learning rates, and acts as regularization."
    },
    {
        "instruction": "What is data augmentation?",
        "input": "",
        "output": "Data augmentation creates variations of training data (rotation, flip, crop for images; synonym replacement for text). Increases effective dataset size and prevents overfitting."
    },
    {
        "instruction": "What is a loss function?",
        "input": "",
        "output": "A loss function measures how well model predictions match ground truth. Examples: MSE for regression, Cross-Entropy for classification. The goal is to minimize the loss."
    },
    {
        "instruction": "What is fine-tuning?",
        "input": "",
        "output": "Fine-tuning continues training a pre-trained model on a specific task or dataset. It adapts the model to new tasks while leveraging knowledge from pre-training."
    },
    {
        "instruction": "What is LoRA (Low-Rank Adaptation)?",
        "input": "",
        "output": "LoRA is a parameter-efficient fine-tuning method that injects trainable low-rank matrices into the model. It reduces trainable parameters by up to 99% while maintaining performance."
    },
    {
        "instruction": "What is PEFT (Parameter-Efficient Fine-Tuning)?",
        "input": "",
        "output": "PEFT is a family of methods (LoRA, Adapters, Prompt Tuning) that fine-tune only a small fraction of parameters, reducing memory and computational requirements."
    },
    {
        "instruction": "What is the chain rule in backpropagation?",
        "input": "",
        "output": "The chain rule is used in backpropagation to compute gradients of the loss with respect to each weight by multiplying gradients layer by layer from output to input."
    },
    {
        "instruction": "What is gradient clipping?",
        "input": "",
        "output": "Gradient clipping prevents exploding gradients by capping gradient values. It's essential for training RNNs and deep networks to maintain training stability."
    },
    {
        "instruction": "What is early stopping?",
        "input": "",
        "output": "Early stopping monitors validation loss and stops training when it stops improving. It prevents overfitting and saves time by avoiding unnecessary epochs."
    },
    {
        "instruction": "What is a learning rate scheduler?",
        "input": "",
        "output": "A learning rate scheduler adjusts the learning rate during training. Common types: Step Decay, Exponential Decay, Cosine Annealing, and Reduce on Plateau."
    },
    {
        "instruction": "What is a confusion matrix?",
        "input": "",
        "output": "A confusion matrix shows prediction results with True Positives, True Negatives, False Positives, and False Negatives. It's used to visualize classification performance."
    },
    {
        "instruction": "What is accuracy in machine learning?",
        "input": "",
        "output": "Accuracy = (Correct Predictions) / (Total Predictions). It's simple but can be misleading for imbalanced datasets where other metrics like precision and recall are better."
    },
    {
        "instruction": "What is the F1 score?",
        "input": "",
        "output": "The F1 score is the harmonic mean of precision and recall: 2 * (Precision * Recall) / (Precision + Recall). It's useful for imbalanced datasets."
    },
    {
        "instruction": "What is ROC-AUC?",
        "input": "",
        "output": "ROC-AUC measures the area under the ROC curve, showing the trade-off between True Positive Rate and False Positive Rate. Higher AUC means better performance."
    },
    {
        "instruction": "What is a neural network layer?",
        "input": "",
        "output": "A neural network layer is a group of neurons that process inputs. Types: Input Layer (receives data), Hidden Layers (process information), Output Layer (produces predictions)."
    },
    {
        "instruction": "What is a neuron in deep learning?",
        "input": "",
        "output": "A neuron (or unit) computes y = activation(sum(weights * inputs) + bias). It's the basic building block of neural networks, inspired by biological neurons."
    },
    {
        "instruction": "What is weight initialization?",
        "input": "",
        "output": "Weight initialization sets initial values for network weights. Proper initialization (Xavier, He) helps training converge faster by preventing vanishing/exploding gradients."
    },
    {
        "instruction": "What is a deep neural network?",
        "input": "",
        "output": "A deep neural network has multiple hidden layers (typically >3). Deep networks can learn hierarchical features, with earlier layers learning simple patterns and later layers complex ones."
    },
    {
        "instruction": "What is the vanishing gradient problem?",
        "input": "",
        "output": "The vanishing gradient problem occurs when gradients become extremely small in deep networks, preventing early layers from learning. Solutions include ReLU and residual connections."
    },
    {
        "instruction": "What is the exploding gradient problem?",
        "input": "",
        "output": "The exploding gradient problem occurs when gradients grow exponentially in deep networks. Solutions: gradient clipping, careful weight initialization, and batch normalization."
    },
    {
        "instruction": "What is a residual connection?",
        "input": "",
        "output": "A residual connection adds the input to the output of a layer: y = f(x) + x. It helps with gradient flow, enabling very deep networks. Used in ResNet and Transformers."
    },
    {
        "instruction": "What is batch size in training?",
        "input": "",
        "output": "Batch size is the number of samples processed before updating weights. Small batch sizes (1-32): more noise, better generalization. Large batch sizes (256+): faster training."
    },
    {
        "instruction": "What is an epoch in machine learning?",
        "input": "",
        "output": "An epoch is one complete pass through the entire training dataset. Multiple epochs (1-100) are typically needed for the model to converge to a good solution."
    }
]

print("Generating 500+ training examples...")

augmented_data = []

for item in ML_DL_DATA:
    augmented_data.append(item.copy())
    
    templates = [
        "Explain {concept}",
        "What is {concept}?",
        "Define {concept}",
        "Tell me about {concept}",
        "Can you explain {concept}?"
    ]
    
    for template in templates:
        words = item["instruction"].split()
        concept = words[-1] if len(words) > 1 else "this concept"
        new_instruction = template.format(concept=concept)
        
        variation = {
            "instruction": new_instruction,
            "input": item["input"],
            "output": item["output"]
        }
        augmented_data.append(variation)
        if len(augmented_data) >= 500:
            break
    if len(augmented_data) >= 500:
        break

print(f"Generated {len(augmented_data)} examples")

output_file = "ml_dl_training_data.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for item in augmented_data:
        json.dump(item, f, ensure_ascii=False)
        f.write("\n")

print(f"Dataset saved to: {output_file}")
print(f"Total examples: {len(augmented_data)}")
print("DATA PREPARATION COMPLETE")

Generating 500+ training examples...
Generated 306 examples
Dataset saved to: ml_dl_training_data.jsonl
Total examples: 306
DATA PREPARATION COMPLETE


<div style="background-color: lightblue; color: black; padding: 10px; border-radius: 5px;">

### Data Loading - Load and Prepare Dataset
</div>

In [4]:
# Cell 2: Data Loading - Load and Prepare Dataset
"""
This cell loads the generated JSONL data, handles UTF-8 encoding issues,
and splits the data into train, validation, and test sets.
"""

import json
from datasets import Dataset
import os

file_path = "ml_dl_training_data.jsonl"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"File {file_path} not found. Please run Cell 1 first.")

print(f"Loading data from: {file_path}")

try:
    with open(file_path, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f]
    print(f"Successfully loaded {len(data)} examples")
except UnicodeDecodeError as e:
    print(f"UnicodeDecodeError: {e}")
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        data = [json.loads(line) for line in f]
    print(f"Loaded {len(data)} examples with error handling")

SUBSET_SIZE = 300

if len(data) > SUBSET_SIZE:
    data = data[:SUBSET_SIZE]
    print(f"Using first {SUBSET_SIZE} examples for training")

print(f"Total examples: {len(data)}")

print("Creating Hugging Face Dataset...")
dataset = Dataset.from_list(data)
print(f"Dataset created with {len(dataset)} examples")

print("Splitting dataset...")

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))

train_dataset = dataset.select(range(train_size))
temp_dataset = dataset.select(range(train_size, len(dataset)))
val_dataset = temp_dataset.select(range(val_size))
test_dataset = temp_dataset.select(range(val_size, len(temp_dataset)))

print(f"  Train: {len(train_dataset)} examples")
print(f"  Validation: {len(val_dataset)} examples")
print(f"  Test: {len(test_dataset)} examples")

print("Saving splits...")
train_dataset.to_json("train_data.jsonl")
val_dataset.to_json("val_data.jsonl")
test_dataset.to_json("test_data.jsonl")

print("Saved train/val/test splits")
print("DATA LOADING COMPLETE")

Loading data from: ml_dl_training_data.jsonl
Successfully loaded 306 examples
Using first 300 examples for training
Total examples: 300
Creating Hugging Face Dataset...
Dataset created with 300 examples
Splitting dataset...
  Train: 240 examples
  Validation: 30 examples
  Test: 30 examples
Saving splits...


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved train/val/test splits
DATA LOADING COMPLETE


<div style="background-color: lightblue; color: black; padding: 10px; border-radius: 5px;">

### Model Setup - Load Model and Configure LoRA
</div>

In [5]:
# Cell 3: Model Setup - Load Model and Configure LoRA
"""
This cell loads the Qwen model, sets up 4-bit quantization for memory efficiency,
and configures LoRA with ultra-fast settings (r=4, max_length=128).
The cell also handles HF_TOKEN authentication for faster downloads.
"""

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
import os

# ============================================
# 1. SET HF_TOKEN FROM FILE
# ============================================

TOKEN_FILE = "D:/edu_laptop/Ironhack_AI_&_DataScience/Huggingface_key.txt"

try:
    with open(TOKEN_FILE, "r", encoding="utf-8") as f:
        hf_token = f.read().strip()
    os.environ["HF_TOKEN"] = hf_token
    print(f"HF_TOKEN loaded from: {TOKEN_FILE}")
except FileNotFoundError:
    print(f"Warning: Token file not found at {TOKEN_FILE}")
    print("Continuing without token (rate limits may apply)")
    hf_token = None
except Exception as e:
    print(f"Error loading token: {e}")
    hf_token = None

# ============================================
# 2. CHECK DEVICE
# ============================================

print("Checking device...")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple MPS (Metal Performance Shaders)")
else:
    device = torch.device("cpu")
    print("Using CPU (training will be very slow)")
    print("Consider using Google Colab with GPU for faster training")

# ============================================
# 3. LOAD MODEL AND TOKENIZER
# ============================================

print("Loading model and tokenizer...")

model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    padding_side="right"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded with vocab size: {tokenizer.vocab_size}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16
)

print(f"Model loaded with {sum(p.numel() for p in model.parameters()):,} parameters")

# ============================================
# 4. PREPARE FOR TRAINING
# ============================================

print("Preparing model for training...")

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# ============================================
# 5. LORA CONFIGURATION (ULTRA-FAST)
# ============================================

print("Setting up LoRA configuration (ULTRA-FAST)...")

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable_params:,} ({trainable_params/total_params*100:.4f}% of {total_params:,} total)")

# ============================================
# 6. LOAD DATASETS
# ============================================

print("Loading datasets...")

train_dataset = load_dataset("json", data_files="train_data.jsonl", split="train")
val_dataset = load_dataset("json", data_files="val_data.jsonl", split="train")

print(f"Train: {len(train_dataset)} examples")
print(f"Validation: {len(val_dataset)} examples")

# ============================================
# 7. TOKENIZE DATASET
# ============================================

print("Tokenizing datasets (max_length=128)...")

def tokenize_function(examples):
    texts = []
    for instruction, output in zip(examples['instruction'], examples['output']):
        text = f"### Instruction:\n{instruction}\n\n### Response:\n{output}"
        texts.append(text)
    
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt"
    )
    
    return {
        "input_ids": tokenized["input_ids"].squeeze(),
        "attention_mask": tokenized["attention_mask"].squeeze()
    }

train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=val_dataset.column_names)

print("Tokenization complete")

# ============================================
# 8. DATA COLLATOR
# ============================================

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator ready")

# ============================================
# 9. SUMMARY
# ============================================

print("="*60)
print("CONFIGURATION SUMMARY (ULTRA-FAST)")
print("="*60)
print(f"""
Model: {model_name}
Device: {device}
LoRA Rank: {lora_config.r}
LoRA Alpha: {lora_config.lora_alpha}
Max Length: 128
Trainable Parameters: {trainable_params:,} ({trainable_params/total_params*100:.4f}%)
Training Data: {len(train_dataset)} examples
Validation Data: {len(val_dataset)} examples
Expected Training Time: 5-8 minutes
""")

print("MODEL SETUP COMPLETE")

HF_TOKEN loaded from: D:/edu_laptop/Ironhack_AI_&_DataScience/Huggingface_key.txt
Checking device...
Using CPU (training will be very slow)
Consider using Google Colab with GPU for faster training
Loading model and tokenizer...
Tokenizer loaded with vocab size: 151643


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\bitsandbytes\backends\default\ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

Generating train split: 0 examples [00:00, ? examples/s]

Train: 240 examples
Validation: 30 examples
Tokenizing datasets (max_length=128)...


Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Tokenization complete
Data collator ready
CONFIGURATION SUMMARY (ULTRA-FAST)

Model: Qwen/Qwen2.5-0.5B
Device: cpu
LoRA Rank: 4
LoRA Alpha: 8
Max Length: 128
Trainable Parameters: 172,032 (0.0546%)
Training Data: 240 examples
Validation Data: 30 examples
Expected Training Time: 5-8 minutes

MODEL SETUP COMPLETE


<div style="background-color: lightblue; color: black; padding: 10px; border-radius: 5px;">

### Training - Fine-Tune the Model
</div>

In [ ]:
# Cell 4: Training - Fine-Tune the Model
"""
This cell performs the actual fine-tuning using the Hugging Face Trainer.
Training is optimized for speed with 1 epoch, batch size 16, and minimal logging.
Expected training time: 5-8 minutes on a GPU.
"""

from transformers import TrainingArguments, Trainer
import time

print("Setting up training arguments (ULTRA-FAST)...")

timestamp = time.strftime("%Y%m%d_%H%M%S")
output_dir = f"./qwen-ml-assistant-{timestamp}"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_steps=10,
    logging_steps=20,
    save_steps=100,
    save_total_limit=1,
    fp16=True,
    gradient_checkpointing=True,
    dataloader_pin_memory=False,
    report_to="none",
)

print("Training arguments ready")

print("Creating trainer...")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("Trainer created")

print("="*60)
print("STARTING TRAINING (ULTRA-FAST)")
print("="*60)
print(f"""
Training Configuration:
- Epochs: {training_args.num_train_epochs}
- Batch Size: {training_args.per_device_train_batch_size}
- Learning Rate: {training_args.learning_rate}
- Max Length: 128
- Training Data: {len(train_dataset)} examples
- Expected Time: 5-8 minutes
""")

try:
    trainer.train()
    print("Training completed successfully")
except Exception as e:
    print(f"Training failed: {e}")
    raise

print("Saving model...")

adapter_path = f"{output_dir}/adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to: {adapter_path}")

print("Merging and saving full model...")
merged_model = model.merge_and_unload()
merged_path = f"{output_dir}/merged"
merged_model.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)
print(f"Full model saved to: {merged_path}")

print("Evaluating model...")
eval_results = trainer.evaluate()
print("Evaluation Results:")
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}")

print("="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"""
Model saved to: {output_dir}
Training Steps: {trainer.state.global_step}
Best Loss: {trainer.state.best_metric:.4f}
Total Training Time: 5-8 minutes
""")

Setting up training arguments (ULTRA-FAST)...
Training arguments ready
Creating trainer...
Trainer created
STARTING TRAINING (ULTRA-FAST)

Training Configuration:
- Epochs: 1
- Batch Size: 16
- Learning Rate: 0.0003
- Max Length: 128
- Training Data: 240 examples
- Expected Time: 5-8 minutes



c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:80: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


<div style="background-color: lightblue; color: black; padding: 10px; border-radius: 5px;">

### Inference Test - Test the Fine-Tuned Model
</div>

In [ ]:
# Cell 5: Inference Test - Test the Fine-Tuned Model
"""
This cell loads the fine-tuned model and tests it with sample questions.
Compares the model's responses to verify that fine-tuning was successful.
"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os

def find_latest_model():
    dirs = [d for d in os.listdir(".") if d.startswith("qwen-ml-assistant-")]
    if not dirs:
        return None
    return sorted(dirs)[-1]

model_dir = find_latest_model()
if not model_dir:
    print("No model found. Please run Cell 4 first.")
    exit()

model_path = f"./{model_dir}/merged"
print(f"Using model: {model_path}")

print("Loading model...")

finetuned_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
    device_map="auto"
)
finetuned_tokenizer = AutoTokenizer.from_pretrained(model_path)

print("Model loaded")

def generate_response(instruction, max_length=100):
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inputs = finetuned_tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(finetuned_model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = finetuned_model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=finetuned_tokenizer.eos_token_id
        )
    
    response = finetuned_tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("### Response:\n")[-1].strip()
    return response

test_questions = [
    "What is machine learning?",
    "Explain gradient descent",
    "What is overfitting?"
]

print("="*60)
print("TESTING FINE-TUNED MODEL")
print("="*60)

for i, question in enumerate(test_questions, 1):
    print(f"Question {i}: {question}")
    print("-" * 40)
    response = generate_response(question)
    print(f"Response: {response}")
    print("-" * 40)

print("INFERENCE COMPLETE")

<div style="background-color: lightblue; color: black; padding: 10px; border-radius: 5px;">

### Speed Comparison Summary 
</div>

<div style="background-color: lightblue; color: black; padding: 10px; border-radius: 5px;">

### Export for Ollama (Optional)
</div>

In [ ]:
# Cell 7: Export for Ollama
"""
This cell creates a Modelfile for Ollama deployment.
The model can be used with Ollama after converting to GGUF format.
"""

import os
import subprocess

print("Creating Modelfile for Ollama...")

def find_latest_model():
    dirs = [d for d in os.listdir(".") if d.startswith("qwen-ml-assistant-")]
    if not dirs:
        return None
    return sorted(dirs)[-1]

model_dir = find_latest_model()
if not model_dir:
    print("No model found. Please run Cell 4 first.")
    exit()

model_path = f"./{model_dir}/merged"
print(f"Using model: {model_path}")

modelfile = f"""
FROM {model_path}

TEMPLATE \"\"\"
### Instruction:
{{{{ .Prompt }}}}

### Response:
\"\"\"

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER stop "###"

SYSTEM \"\"\"
You are an expert AI/ML assistant specialized in deep learning, 
machine learning, and data science. Provide accurate, 
educational, and clear explanations.
\"\"\"
"""

with open("Modelfile", "w") as f:
    f.write(modelfile)

print("Modelfile created")

print("""
To deploy with Ollama:
1. Install Ollama from https://ollama.ai
2. Convert model to GGUF format using llama.cpp
3. Run: ollama create ml-assistant -f Modelfile
4. Run: ollama run ml-assistant "What is machine learning?"
""")

print("EXPORT PREPARATION COMPLETE")

<div style="background-color: lightblue; color: black; padding: 10px; border-radius: 5px;">

### Export for Ollama (Optional)
</div>